
# Regression Dataset Tutorial: GNN Training

This notebook demonstrates how to set up and train a Graph Neural Network (GNN) using PyTorch and PyTorch Geometric. You will learn how to prepare data, define a model, and train and evaluate it on a dataset. Let's get started!


In [1]:
# Loading the Type1 datasets
import torch
from torch_geometric.loader import DataLoader
from copy import deepcopy

# Custom imports
from Type1 import Type1
from Type2 import Type2
from hg_models import HeteroGNN_SAGE
from early_stopping import EarlyStopping


## Loading the Dataset

In this section, we load various datasets using the `Type2` class. After that we can see the number of nodes os `type2_dataset1` and the structure of one of his nodes


In [ ]:
type2_dataset1 = Type2("tmp", "../data" , "../y_labels", "rdf")
type2_dataset2 = Type2("tmp", "../data" , "../y_labels", "dubbo")
type2_dataset3 = Type2("tmp", "../data" , "../y_labels", "H2")
type2_dataset4 = Type2("tmp", "../data" , "../y_labels", "hadoop")
type2_dataset5 = Type2("tmp", "../data" , "../y_labels", "systemds")
type2_dataset6 = Type2("tmp", "../data" , "../y_labels", "ossbuilds")

In [3]:
type2_dataset1

Type2(478)

In [4]:
type2_dataset1[0]

HeteroData(
  y=[1],
  code_structure={ x=[8, 73] },
  literals_and_constants={ x=[24, 73] },
  declarations={ x=[8, 73] },
  types_and_references={ x=[5, 73] },
  control_flow={ x=[1, 73] },
  expressions_and_operations={ x=[2, 73] },
  (code_structure, 0, code_structure)={ edge_index=[2, 20] },
  (code_structure, 0, literals_and_constants)={ edge_index=[2, 7] },
  (literals_and_constants, 0_rev, code_structure)={ edge_index=[2, 7] },
  (literals_and_constants, 0, code_structure)={ edge_index=[2, 7] },
  (code_structure, 0_rev, literals_and_constants)={ edge_index=[2, 7] },
  (code_structure, 0, declarations)={ edge_index=[2, 3] },
  (declarations, 0_rev, code_structure)={ edge_index=[2, 3] },
  (declarations, 0, code_structure)={ edge_index=[2, 3] },
  (code_structure, 0_rev, declarations)={ edge_index=[2, 3] },
  (declarations, 0, declarations)={ edge_index=[2, 28] },
  (declarations, 0, literals_and_constants)={ edge_index=[2, 9] },
  (literals_and_constants, 0_rev, declarations)={

In [5]:
# Training and testing functions for the model
def train(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data in data_loader:
        data = data.to(device)
        optimizer.zero_grad()
        graph_output = model(data.x_dict, data.edge_index_dict, data.batch_dict)
        target = data.y.view(-1, 1).to(device)
        loss = criterion(graph_output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)

@torch.no_grad()
def test(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds, reals = [], []
    for data in loader:
        data = data.to(device)
        
        out = model(data.x_dict, data.edge_index_dict, data.batch_dict)
        loss = criterion(out, torch.reshape(data.y, (data.y.shape[0], 1)))
        total_loss += float(loss)

        pred = out.detach().cpu().numpy()[:, 0] if out.dim() > 1 else out.detach().cpu().numpy()
        real = data.y.detach().cpu().numpy()
        preds.extend(pred)
        reals.extend(real)

    return total_loss / len(loader.dataset), 0, preds, reals


## Defining Hyperparameters

Here we set the model's hyperparameters like the device (CPU or GPU), patience (for early stopping), hidden channels, batch size, and learning rate. You can adjust these parameters based on your computational resources and dataset size.


In [6]:
# Defining the device, patience, hidden channels, batch size, and learning rate
device = "cpu" # cuda
patience = 15
hidden_channels = 32
batch_size = 32
learning_rate = 0.01

## Training the Model

In this section, we define the model, optimizer, and loss criterion, and set up the data loaders for training, validation, and testing. We use a special `collate_function` which is available in all the `Type2` datasets (normally the default one works just fine). The model is trained using the Adam optimizer and `L1Loss` as the loss function. We also use early stopping to prevent overfitting by monitoring the validation loss. The best model based on validation loss is saved, and training progress is printed at regular intervals.


In [7]:
model = HeteroGNN_SAGE(type2_dataset1, hidden_channels=hidden_channels, out_channels=1, num_layers=4, device=device).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.L1Loss()

In [8]:
# Training and testing functions for the model
train_loader = DataLoader(type2_dataset1.load_split("train"), batch_size=batch_size, shuffle=True, collate_fn=type2_dataset1.hetero_collate)
val_loader = DataLoader(type2_dataset1.load_split("val"), batch_size=batch_size, shuffle=False, collate_fn=type2_dataset1.hetero_collate)
test_loader = DataLoader(type2_dataset1.load_split("test"), batch_size=batch_size, shuffle=False, collate_fn=type2_dataset1.hetero_collate)

In [9]:
# Training and testing functions for the model
early_stopping = EarlyStopping(patience=patience)
best_model_state = None
best_val_loss = float('inf')

for epoch in range(1, 101):
    # Train the model
    train_loss = train(model, train_loader, optimizer, criterion, device)

    # Validate the model
    val_loss, _, _, _ = test(model, val_loader, criterion, device)

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = deepcopy(model.state_dict())
    
    # Print progress every 10 epochs or at the first epoch
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Val Loss: {val_loss:.3f}')

    # Early stopping check based on validation loss
    if early_stopping(val_loss):
        print(f'Early stopping at epoch {epoch}')
        break

Epoch: 001, Train Loss: 604.422, Val Loss: 0.186
Epoch: 010, Train Loss: 1.613, Val Loss: 0.015
Epoch: 020, Train Loss: 0.527, Val Loss: 0.011
Epoch: 030, Train Loss: 0.233, Val Loss: 0.007
Early stopping triggered after 15 epochs of no improvement.
Early stopping at epoch 30



## Testing the Model

The `test` function evaluates the model's performance on the validation or test dataset. It runs in evaluation mode, calculating the loss without performing backpropagation, and collects predictions for further analysis.


In [12]:
# Training and testing functions for the model
# Reload the best model after training
model.load_state_dict(best_model_state)

# After training completes, evaluate the model on the test set
test_loss, _, predt, realt = test(model, test_loader, criterion, device)
val_loss, _, predt, realt = test(model, val_loader, criterion, device)

# Print final test results
print(f'Final Test Loss: {test_loss:.3f}, Final val Loss: {val_loss:.3f}')

Final Test Loss: 0.008, Final val Loss: 0.007


In [ ]:
# Loading the Type1 datasets
type1_dataset1 = Type1("tmp", "../data" , "../y_labels", "rdf")
type1_dataset2 = Type1("tmp", "../data" , "../y_labels", "dubbo")
type1_dataset3 = Type1("tmp", "../data" , "../y_labels", "H2")
type1_dataset4 = Type1("tmp", "../data" , "../y_labels", "hadoop")
type1_dataset5 = Type1("tmp", "../data" , "../y_labels", "systemds")
type1_dataset6 = Type1("tmp", "../data" , "../y_labels", "ossbuilds")